# Fellegi-Sunter vs Non-ID — Champion / Challenger

Challenger = an **unlabelled Splink Fellegi-Sunter** record-linkage model (FS-A no-TF, FS-B +term-frequency). Champion = the **Non-ID** tiered exact-match rule engine. Both score the **same** Splink-blocked holdout pairs; ground truth (`DW_ID_Key`) is joined **only after scoring**.

**Configure the whole run in the RUN PARAMETERS cell below** — fit / holdout sizes, seed, target precision, variants. Set `USE_SYNTHETIC=False` on the platform to read the real overlap via Omni Hub.

In [1]:
# ============================================================================
# RUN PARAMETERS  —  edit these, then Run All
# ============================================================================
USE_SYNTHETIC    = True        # True = offline synthetic data; False = real data via Omni Hub (platform)

FIT_ROWS         = 1_000       # records the FS model is FIT on (train).      N rows = N/2 entities (Strive+TU)
HOLDOUT_ROWS     = 2_000       # records SCORED + evaluated (the held-out 'score' set). N rows = N/2 entities
SEED             = 42          # deterministic sample draw

TARGET_PRECISION = 0.99        # FS operating point for the champion/challenger head-to-head
VARIANTS         = ('FS-A', 'FS-B')   # use ('FS-A',) for a quicker single-variant run
DETAIL_VARIANT   = 'FS-B'      # which variant the detailed diagnostics show
# ----------------------------------------------------------------------------
# FIT_ROWS / HOLDOUT_ROWS must be positive EVEN numbers. Bigger = slower but more robust.
#   quick smoke: FIT_ROWS=500,  HOLDOUT_ROWS=400      fuller run: 10_000 / 20_000
# ============================================================================

import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path.cwd().parent))   # project root on the path

from fs_cc import FellegiSunterConfig, runner, diagnostics as diag, champion_challenger as cc, champion
from fs_cc.theme import register_omni_theme, banner
from IPython.display import display, HTML
register_omni_theme()

cfg = FellegiSunterConfig(
    sample_sizes=(FIT_ROWS,),
    released_sizes=(FIT_ROWS,),
    seeds=(SEED,),
    holdout_rows=HOLDOUT_ROWS,
    variants=VARIANTS,
)
cfg.validate()
display(banner('Fellegi-Sunter Champion / Challenger',
               f"{'synthetic' if USE_SYNTHETIC else 'platform (Omni Hub)'} · fit {FIT_ROWS:,} rows · "
               f"holdout {HOLDOUT_ROWS:,} rows · variants {VARIANTS}"))

## 1 · Load data

Synthetic (record-level, fabricated) locally; the real TransUnion×Strive overlap via Omni Hub on-platform.

In [2]:
if USE_SYNTHETIC:
    fit, holdout = runner.load_synthetic_cohorts(cfg)
else:
    # Platform path — wrap the Omni Hub magic in a query_runner(sql) -> DataFrame
    get_ipython().run_line_magic('load_ext', 'omni_hub')
    def query_runner(sql):
        return get_ipython().run_line_magic('omni_hub', sql).DataFrame()
    fit, holdout = runner.load_real_cohorts(query_runner, cfg)

print('fit   :', fit.manifest['row_count'], 'rows /', fit.manifest['entity_count'], 'entities')
print('holdout:', holdout.manifest['row_count'], 'rows /', holdout.manifest['entity_count'], 'entities')
print('null rates (fit):', {k: round(v, 3) for k, v in fit.manifest['null_rates'].items()})

fit   : 1000 rows / 500 entities
holdout: 2000 rows / 1000 entities
null rates (fit): {'first_name_key': 0.027, 'surname_key': 0.03, 'dob_key': 0.025, 'email_key': 0.026, 'phone_key': 0.027}


## 2 · Pre-fit EDA — the comparison space (label-free)

What the inputs look like before any model is trusted: completeness, cardinality (the placeholder-hash gate), and the term-frequency tail that motivates FS-B.

In [3]:
diag.field_completeness_fig(fit.matcher, cfg).show()
display(diag.field_cardinality_table(fit.matcher, cfg))
diag.value_frequency_figs(fit.matcher, cfg)['surname_key'].show()

,field,distinct_values,non_null,null_rate,top_value_share,mean_value_freq
0,first_name_key,68,973,0.027,0.0483,14.31
1,surname_key,105,970,0.030,0.0299,9.24
2,dob_key,208,975,0.025,0.0144,4.69
3,email_key,630,974,0.026,0.0021,1.55
4,phone_key,590,973,0.027,0.0021,1.65


## 3 · Fit FS-A + FS-B (unlabelled) and run the experiment

No labels are used in fitting. `run_experiment` fits each variant, scores the holdout, joins the sealed oracle, runs the Non-ID champion on the same pairs, and builds the head-to-head + leaderboard.

In [4]:
exp = runner.run_experiment(cfg, fit, holdout, target_precision=TARGET_PRECISION)
print('Leaderboard (ranked by PR-AUC):')
display(exp['leaderboard'])
det = exp['results'][DETAIL_VARIANT]
print('operating threshold:', round(det['threshold'], 4),
      '| fit seconds:', round(det['fit_result'].timings['total_seconds'], 1))

You are using the default value for `max_pairs`, which may be too small and thus lead to inaccurate estimates for your model's u-parameters. Consider increasing to 1e8 or 1e9, which will result in more accurate estimates, but with a longer run time.


You are using the default value for `max_pairs`, which may be too small and thus lead to inaccurate estimates for your model's u-parameters. Consider increasing to 1e8 or 1e9, which will result in more accurate estimates, but with a longer run time.


Leaderboard (ranked by PR-AUC):


,contestant,pr_auc,auroc,precision,recall,f1,predicted_positive
0,FS-A,0.996498,0.999996,0.990816,0.971,0.980808,980
1,FS-B,0.996365,0.999995,0.990702,0.959,0.974593,968
2,champion (Non-ID),0.989285,0.995997,0.819835,0.992,0.897738,1210
3,B1 (agreement count),0.979804,0.999960,0.020186,1.000,0.039573,49539
4,B0 (deterministic),0.969037,0.985998,0.996923,0.972,0.984304,975


operating threshold: 0.8101 | fit seconds: 3.9


## 4 · FS model & fit diagnostics (label-free)

The match-weight scorecard **is** the weight-of-evidence model, learned without labels. The prior λ is rank-invariant (it moves the threshold/calibration, not the ranking).

In [5]:
display(HTML(diag.parameter_validity_html(det['fit_result'])))
display(diag.prior_summary(det['fit_result'].model_settings))
diag.mu_parameters_fig(det['fit_result'].parameters).show()
diag.match_weight_scorecard_fig(det['fit_result'].parameters).show()
diag.match_weight_histogram_fig(det['fit_result'].holdout_predictions).show()
display(diag.tf_effect_table(det['fit_result']))

,quantity,value
0,probability_two_random_records_match (λ),0.002794
1,prior match weight (bits),-8.479300


,field,term_count,minimum_u_value,affected_term_count,affected_term_rate
0,first_name_key,68,0.00001,0,0.0
1,surname_key,105,0.00001,0,0.0
2,dob_key,208,0.00001,0,0.0


## 5 · Scoring & quality (sealed oracle)

Everything below joins `DW_ID_Key` **after** scoring. FS probabilities are poorly calibrated on a stratified cohort — trust the rank/weight, not the absolute probability. Aggregate PR-AUC/AUROC run high because the quality view is dominated by easy negatives; read the per-pattern precision + threshold-matched table for the real story.

In [6]:
display(HTML(diag.kpi_tiles_html(det['evaluation'].outcomes)))
diag.score_by_label_fig(det['labelled'], 'match_weight', is_weight=True).show()
diag.pr_curve_fig(det['labelled']).show()
diag.threshold_fig(det['evaluation'].threshold_table).show()
diag.calibration_fig(det['labelled']).show()
diag.lift_decile_fig(det['labelled']).show()
display(det['evaluation'].agreement_pattern_table)

,agreement_pattern,candidate_pairs,true_matches,mean_match_probability,observed_precision
0,first_name_key:M|surname_key:M|dob_key:M|email...,329,329,1.000000e+00,1.0
1,first_name_key:M|surname_key:M|dob_key:M|email...,34,34,9.999971e-01,1.0
2,first_name_key:M|surname_key:M|dob_key:M|email...,87,87,9.999824e-01,1.0
3,first_name_key:M|surname_key:M|dob_key:M|email...,22,22,9.999971e-01,1.0
4,first_name_key:M|surname_key:M|dob_key:M|email...,2,2,9.989101e-01,1.0
...,...,...,...,...,...
155,first_name_key:X|surname_key:X|dob_key:X|email...,2,2,3.401936e-01,1.0
156,first_name_key:X|surname_key:X|dob_key:X|email...,2526,0,2.105776e-06,0.0
157,first_name_key:X|surname_key:X|dob_key:X|email...,45197,0,3.837739e-07,0.0
158,first_name_key:X|surname_key:X|dob_key:X|email...,51854,0,6.273030e-07,0.0


## 6 · Explainability — per-pair match-weight waterfalls

Prior + each field's contribution → the pair's total match weight (the FS SHAP-equivalent). Shows weight contributions, not raw keys (PII-safe).

In [7]:
wf = cc.waterfall_examples(det['labelled'], cfg, det['fit_result'].model_settings, det['threshold'])
for label, fig in wf.items():
    print(label); fig.show()

win (true match, FS correct)


false positive (FS says match, isn't)


false negative (true match, FS misses)


## 7 · Champion vs Challenger

The Non-ID rules (champion) vs Fellegi-Sunter (challenger) on the **same** pairs. The champion is discrete, so the fair comparison is threshold-matched.

In [8]:
a = det['analysis']
display(HTML(cc.cc_kpis_html(a)))
cc.cc_agreement_fig(a).show()
cc.cc_lift_fig(a).show()
print('Threshold-matched:'); display(cc.cc_threshold_matched(a))
cc.cc_score_binned_fig(a).show()
champion.nonid_tier_fig(champion.nonid_tier_summary(det['per_pair'])).show()
cc.leaderboard_fig(exp['leaderboard']).show()
print('FS-A vs FS-B (term-frequency verdict input):'); display(exp['variant_delta'])

Threshold-matched:


,basis,precision,recall,note
0,champion (NonID),0.8198,0.992,its operating point
1,FS @ NonID precision (0.820),0.8201,0.994,thr 0.062 — recall uplift +0.002
2,FS @ NonID recall (0.992),0.8881,0.992,thr 0.107 — precision uplift +0.068


FS-A vs FS-B (term-frequency verdict input):


,metric,FS-A,FS-B,delta (B-A)
0,pr_auc,0.996498,0.996365,-0.000132
1,auroc,0.999996,0.999995,-0.000001
2,mrr,0.998000,0.998500,0.000500
3,recall_at_precision_90,0.987000,0.991000,0.004000
4,precision_at_recall_90,1.000000,0.997904,-0.002096
5,blocking_recall,1.000000,1.000000,0.000000


## 8 · Build the self-contained HTML report (+ lossless PNG)

One offline file: inlined plotly + vanilla-JS interactivity. The PNG is a byte-identical twin — decode it back with `decode_report_png.py`.

In [9]:
html = runner.build_report(exp, detail_variant=DETAIL_VARIANT)
paths = runner.save_report(html, Path.cwd().parent / 'outputs')
print('report :', paths['html'])
print('png    :', paths['png'], '(', paths['image'], ')')
display(HTML(f"<a href='{paths['html']}' target='_blank'>open the champion/challenger report</a>"))

report : C:\Repos\DS_Platform-graphmatching\epics\graph_matching\projects\fs_champion_challenger_20260723\outputs\fs_champion_challenger_report.html
png    : C:\Repos\DS_Platform-graphmatching\epics\graph_matching\projects\fs_champion_challenger_20260723\outputs\fs_champion_challenger_report.png ( 698x698 )


## Notes / gate memo

- **Hypothesis (UFS1 · Phase-3 flavour):** unlabelled FS, scored on non-ID PII only, matches or beats the Non-ID rule engine on the same pairs (agreement, lift, threshold-matched precision/recall).
- **Read the leaderboard + threshold-matched table** for the verdict; **FS-A vs FS-B** for the term-frequency decision (retain / constrain / drop).
- **Label boundary:** `DW_ID_Key` is never in the matcher; it is joined only after scoring.
- **Interpretation:** aggregate PR-AUC/AUROC are inflated by easy negatives — judge discrimination from the per-agreement-pattern precision and the threshold-matched champion/challenger table.